# Week 2, Day 2: A streaming WrestleMania preview writer

**Lesson:** [week2/day2.ipynb](../../../week2/day2.ipynb), *Gradio: user interfaces for your models*. **About 40 minutes.**

**What you practice here**

- Streaming a reply: a Chat Completions call with streaming on, adding each chunk's text to the text so far and yielding it, so the page grows as the model writes.
- Letting the user pick the model by name, as the lesson's `stream_model` does, with a clear error for an unknown name.
- The lesson's exercise: a UI on a Week 1 generator. Clear the output at once, fetch the page with the course's helper, and stream from the chosen model.
- Building the page with `gr.Interface`: Textboxes and a Dropdown in, a Markdown output, around a streaming function.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import os, pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT is None and ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ):
    # in Colab: fetch the checker from your fork on GitHub, and the packages Colab lacks
    import subprocess, urllib.request
    ROOT = pathlib.Path("/content/fork")
    (ROOT / "plan").mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/starlest/llm_engineering/main/plan/check.py", ROOT / "plan" / "check.py")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv", "tiktoken"], check=False)
    print("In Colab, the checks run here too. To keep your work, use File > Save a copy in GitHub, to this notebook's path in your fork.")
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week2/day2", globals())
else:  # outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here.")

## The task

The lesson's exercise puts a Gradio UI on the Week 1 brochure generator: the page takes a company name, a URL and a
model, and the brochure streams in as the model writes it. Build the same for wrestling fans: give an event's name and
a page about it, pick a model, and a short preview of the event streams in.

`gr.Interface` calls a function with the values of its inputs and shows what it returns; when the function is a
generator, the page shows each value it yields. So the functions behind the page are generators that yield the text
so far, and the last part builds the page around them. The functions that call a model take a client, or a dict of
clients, as their first parameter, and `build_page` takes the gradio module. The checks hand them stand-ins shaped
like the real client and like gradio, so nothing is called, nothing is spent and no page opens.

## What each call should give

```python
list(stream_text(client, "Write a preview of WrestleMania III."))
# makes one request with streaming on, and yields the text so far after each chunk:
# ['', '## WrestleMania', '## WrestleMania III', '## WrestleMania III\n\nHulk Hogan', ...]

MODELS = {"GPT": (OpenAI(), "gpt-4.1-mini"),
          "Llama": (OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"), "llama3.2")}
list(stream_model(MODELS, "Write a preview of WrestleMania III.", "Mistral"))
# ValueError: Unknown model 'Mistral'. Choose from: GPT, Llama

stream_preview(MODELS, fetch_website_contents, "WrestleMania III",
               "https://en.m.wikipedia.org/wiki/WrestleMania_III", "Llama")
# yields '' at once, then fetches the page and yields the preview from llama3.2 as it grows

build_page(gr, respond, ["GPT", "Llama"])
# a gr.Interface around respond: two Textboxes and a Dropdown in, a Markdown out; .launch() opens it
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `SYSTEM`

Tells the model to write a short, lively preview of a wrestling event for fans, in markdown. The model writes it from
the page about the event that it is sent. Like the lesson's system message, ask for markdown without code blocks, so
that the page's Markdown output can show it. The wording is yours: write it as a string of 20 characters or more that
mentions markdown.

<details><summary>Hint</summary>

The lesson's was `"You are a helpful assistant that responds in markdown without code blocks"`. Yours says what to
write: a preview of the event, from the page it is sent.

</details>

In [ ]:
SYSTEM = ""

In [ ]:
check("SYSTEM")

### 2 · `stream_text(client, prompt, model="gpt-4.1-mini")`

Streams one reply, and yields the text so far after each chunk: the lesson's `stream_gpt`, with the client passed in.
Make one Chat Completions call with the model `model`, streaming on, and two messages: a system message holding
`SYSTEM`, then a user message holding `prompt`. For each chunk of the stream, add the chunk's text to the text so far,
and yield the text so far. A chunk's text can be `None` (the last chunk's is), so count it as `""` then, as the
lesson's `or ""` does.

<details><summary>Hint</summary>

```python
stream = client.chat.completions.create(model=model, messages=messages, stream=True)
for chunk in stream:
    result += chunk.choices[0].delta.content or ""
    yield result
```

</details>

In [ ]:
def stream_text(client, prompt, model="gpt-4.1-mini"):
    # Write stream_text here, as described above.
    raise NotImplementedError("stream_text is not written yet")

In [ ]:
check("stream_text")

### 3 · `stream_model(models, prompt, model_name)`

Streams from the model the user picked by name. `models` is a dict from a name shown in the dropdown, such as `"GPT"`
or `"Llama"`, to a `(client, model)` pair. Like the lesson's `stream_model`, it is a generator. If `model_name` is not
a key of `models`, raise `ValueError` with a message that names `model_name` and every choice, and call no client.
Otherwise, yield everything `stream_text` yields for that name's client and model, and call no other client.

<details><summary>Hint</summary>

`client, model = models[model_name]`, then `yield from stream_text(client, prompt, model=model)`.

</details>

In [ ]:
def stream_model(models, prompt, model_name):
    # Write stream_model here, as described above.
    raise NotImplementedError("stream_model is not written yet")

In [ ]:
check("stream_model")

### 4 · `stream_preview(models, fetch, event, url, model_name)`

Writes the preview: it clears the preview box at once, fetches the event's page, then streams the preview from the
chosen model. It is the lesson's `stream_brochure`, for an event. `fetch` is the course's `fetch_website_contents` in
the real run; the checks pass a stand-in. Write it as a generator that:

1. first yields `""`, before it does anything else, so the preview box clears at once, while the web page is being
   fetched;
2. calls `fetch(url)` once, and builds a user prompt that names `event` and holds the whole text that `fetch` returns;
3. then yields everything `stream_model` yields for that prompt and `model_name`.

<details><summary>Hint</summary>

`yield ""` comes first, then `page = fetch(url)`, and last `yield from stream_model(models, prompt, model_name)`.

</details>

In [ ]:
def stream_preview(models, fetch, event, url, model_name):
    # Write stream_preview here, as described above.
    raise NotImplementedError("stream_preview is not written yet")

In [ ]:
check("stream_preview")

### 5 · `build_page(gr, fn, choices)`

Builds the page around `fn`, as the lesson's last `gr.Interface` does, and returns it without launching it. `gr` is
the gradio module (the checks pass a stand-in), `fn` is the function the page calls, and `choices` are the names for
the model dropdown. Make one `gr.Interface` with:

- `fn` as its function, and a title of your own;
- three inputs: a Textbox for the event's name, a Textbox for the page's URL, and a Dropdown of `choices` whose value
  starts as the first choice;
- one output: a Markdown component, where the preview appears;
- one row of examples, of three values: an event, a URL and one of `choices`;
- `flagging_mode="never"`, as in the lesson.

Return the `Interface`, so that Try it for real can call `.launch()` on it.

<details><summary>Hint</summary>

```python
gr.Interface(fn=fn, title=..., inputs=[gr.Textbox(label=...), gr.Textbox(label=...),
                                       gr.Dropdown(choices, label=..., value=choices[0])],
             outputs=[gr.Markdown(label=...)], examples=[[event, url, choices[0]]], flagging_mode="never")
```

</details>

In [ ]:
def build_page(gr, fn, choices):
    # Write build_page here, as described above.
    raise NotImplementedError("build_page is not written yet")

In [ ]:
check("build_page")

## Check everything

The cell below runs all 9 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 2, Day 2 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

A preview on `gpt-4.1-mini` costs a fraction of a cent, and one on Claude Sonnet about a cent. The Llama choice needs
Ollama running, with `llama3.2` pulled.

1. Load your keys with `load_dotenv(override=True)`, and build `MODELS`: `"GPT"` with `OpenAI()` and `gpt-4.1-mini`,
   `"Llama"` with an Ollama client and `llama3.2`, and `"Claude"` with the Anthropic endpoint and
   `claude-sonnet-4-5-20250929` if you have the key.
2. Import the course's `fetch_website_contents` from `week2/scraper.py`, after adding `ROOT / "week2"` to `sys.path`
   (`ROOT`, your fork's folder, comes from the first cell). Print the first 500 characters it gets from
   `https://en.m.wikipedia.org/wiki/WrestleMania_III`, the mobile page: the helper keeps only the first 2,000
   characters of a page's text, and that is what the model will read.
3. Write a generator function `respond(event, url, model_name)` that does
   `yield from stream_preview(MODELS, fetch_website_contents, event, url, model_name)`. Gradio streams only from a
   generator function, so a lambda will not do.
4. Open the page with `build_page(gr, respond, list(MODELS)).launch(inbrowser=True)`, after `import gradio as gr`, and
   ask for a preview of WrestleMania III from its mobile Wikipedia page.

<details><summary>Stuck? One way to write it</summary>

```python
import os
import sys

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
MODELS = {"GPT": (OpenAI(), "gpt-4.1-mini"),
          "Llama": (OpenAI(base_url="http://localhost:11434/v1", api_key="ollama"), "llama3.2")}
if os.getenv("ANTHROPIC_API_KEY"):
    MODELS["Claude"] = (OpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url="https://api.anthropic.com/v1/"),
                        "claude-sonnet-4-5-20250929")

sys.path.insert(0, str(ROOT / "week2"))  # ROOT, your fork's folder, comes from the first cell
from scraper import fetch_website_contents

print(fetch_website_contents("https://en.m.wikipedia.org/wiki/WrestleMania_III")[:500])  # what the model reads first


def respond(event, url, model_name):  # a generator function, so Gradio streams it
    yield from stream_preview(MODELS, fetch_website_contents, event, url, model_name)


build_page(gr, respond, list(MODELS)).launch(inbrowser=True)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.